# Simulated Annealing — Full Hard/Logical/Soft Constraint Implementation

**Scope of this notebook**

- Semesters included: configurable in the cell below (`SEMESTERS`). Currently set to
  **24A, 24B, 25A, 25B**, per request. Change that one list to work with any other
  combination of the six terms available in `schedule_cleaned_it.csv`
  (23A, 23B, 24A, 24B, 25A, 25B) — nothing else in the notebook needs to change.
- Tutor-course-class assignment is **fixed input** (who teaches what is never changed).
  The optimiser only reassigns the **timeslot and room** of each already-assigned session.
- Every constraint from `Objective_Function_and_Constraints.docx` is implemented as its
  **own separate, commented function**, named to match that document's IDs (H1–H7 hard,
  L1 logical, S1–S6 soft), so you can see exactly where each one is checked and exactly
  what it contributes to the cost.

**Important modeling decision, read before changing `SEMESTERS`:**
Each semester is its own independent weekly timetable — `24A`'s "Tuesday 14:00" and
`25B`'s "Tuesday 14:00" are different points in the real calendar that never literally
co-occur, even though they share the same weekly label. Every constraint that depends on
"the same day/time" (room clash, tutor clash, cross-campus back-to-back, idle-gap cap,
and the day-of-week soft preferences) is therefore scored **within each semester
separately**, then combined across the semesters you selected. Only the fairness
function (Section 6) looks at a tutor's *combined* total across every selected semester,
because that is a person-level property, not a per-term one. Getting this scoping wrong
was the first bug caught while building this notebook — see the note after the H2/H3
functions for the concrete numbers.

**Two things this notebook cannot fully do yet, flagged rather than guessed at:**
- **H6 (curriculum/cohort conflict)** — needs a course→curriculum (year+major cohort)
  mapping that does not exist in the current cleaned CSVs. Implemented as an explicit
  placeholder that returns 0, wired into the cost function exactly where it would plug
  in once that mapping exists.
- **16 tutors and 12 rooms** in this 4-semester slice do not appear in
  `tutors_anonymized.csv` / `rooms_capacities.csv`. Documented fallbacks are used
  (see the data-loading section) and the affected IDs are printed so nothing is silently
  papered over.


## 0. Configuration

Everything semester-, weight-, and hyperparameter-related lives here. Change this cell to reuse the notebook for a different set of semesters or to retune the objective.

In [1]:
import csv
import math
import random
import time
from collections import defaultdict

random.seed(42)  # reproducibility, carried over from the small-slice draft

# ---- Which semesters to include in this run ----
# Any subset of the six terms present in schedule_cleaned_it.csv: 23A, 23B, 24A, 24B, 25A, 25B
#SEMESTERS = {"24A", "24B", "25A", "25B"}
SEMESTERS = {"25B"}

# ---- Where the CSVs live ----
# Point this at your local "Cleaned Data" folder. The small-slice draft had this
# hardcoded to a specific Mac path -- that was flagged as needing a fix, this is it.
DATA_DIR = "/Users/narjishubail/Desktop/Thesis/Data/Cleaned Data V2/IT Scheduling Data"

SCHEDULE_CSV = f"{DATA_DIR}/schedule_cleaned_it.csv"
TUTOR_PROFILE_CSV = f"{DATA_DIR}/tutors_it_profiles.csv"
ROOM_CSV = f"{DATA_DIR}/rooms_capacities.csv"

# ---- Objective weights, per Objective_Function_and_Constraints.docx Section 1.3 ----
HARD_PENALTY = 1000      # alpha: weight per hard-constraint violation (H1-H7). Unchanged
                          # from the small-slice draft's HARD_PENALTY.
LOGICAL_PENALTY = 50      # beta: weight per logical-constraint violation (L1). NEW, and
                          # explicitly a placeholder -- not yet calibrated. Deliberately
                          # between alpha (a double-booking is infeasible) and the soft
                          # preference weights (a personal-taste violation), since a
                          # >4h idle gap is an institutional problem, not infeasibility.
SOFT_WEIGHTS = {          # w_k in V_t = sum_k w_k * Penalty_t,k(s). All placeholders of 1,
    "P1": 1, "P2": 1, "P3": 1, "P4": 1, "P5": 1, "P6": 1,   # to be calibrated against the
}                          # tutor-preference survey (RQ1) -- separate, not-yet-done work.

IDLE_CAP_MINUTES = 240    # L1: maximum allowed idle gap, in minutes (4 hours)

# ---- Simulated annealing hyperparameters ----
# T0/cooling carried over unchanged from the small-slice draft (Kirkpatrick, Gelatt &
# Vecchi 1983 classical cooling schedule). MAX_ITERATIONS is raised for the much larger
# search space here (2000+ sessions vs. 87), and is a starting point, not a tuned value --
# formal tuning against runtime/quality tradeoffs is separate future work.
MAX_ITERATIONS = 10000
T0 = 50.0
COOLING = 0.998


## 1. Load data and build the working session list

The small-slice draft matched sessions onto a fixed 35-slot grid from `slots.csv` and
silently dropped anything that didn't fit. That grid only covers seven 2-hour start
times (08:00 to 20:00); the real multi-semester data also has sessions starting at
09:00, 11:00, 13:00, 15:00, 17:00, and 19:00 (about 2% of sessions). To avoid silently
losing real sessions, this notebook builds the slot grid **empirically**, directly
from whatever (day, start time) combinations actually appear in the selected semesters'
data. All sessions in this dataset are a fixed 120-minute duration, verified directly
against the data before writing this.

In [2]:
def load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

schedule_rows_all = load_csv(SCHEDULE_CSV)
tutor_profile_rows = load_csv(TUTOR_PROFILE_CSV)
room_rows = load_csv(ROOM_CSV)

tutor_to_profile_raw = {r["TUTOR_ID"]: r["PROFILE_ID"] for r in tutor_profile_rows}
room_capacity = {r["ROOM_ID"]: int(r["CAPACITY"]) for r in room_rows}
room_building = {r["ROOM_ID"]: r["BUILDING_ID"] for r in room_rows}

# ---- filter to the selected semesters ----
schedule_rows = [r for r in schedule_rows_all if r["SEMESTER"] in SEMESTERS]
print(f"Loaded {len(schedule_rows_all)} total sessions across all semesters; "
      f"{len(schedule_rows)} sessions in the selected semesters {sorted(SEMESTERS)}.")


Loaded 3216 total sessions across all semesters; 556 sessions in the selected semesters ['25B'].


In [3]:
def to_minutes(hhmm):
    h, m = map(int, hhmm.split(":"))
    return h * 60 + m

DAY_ORDER = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

# ---- empirical slot registry: every (day, start_time) actually used, in day/time order ----
slot_keys = sorted(
    set((r["DAY"], r["START_TIME"]) for r in schedule_rows),
    key=lambda k: (DAY_ORDER.index(k[0]), to_minutes(k[1])),
)
slot_id_of = {}   # (day, start_time) -> slot_id
slot_info = {}    # slot_id -> (day, start_minute, end_minute)
for i, (day, start) in enumerate(slot_keys, start=1):
    sid = f"SLOT{i:03d}"
    slot_id_of[(day, start)] = sid
    start_min = to_minutes(start)
    slot_info[sid] = (day, start_min, start_min + 120)  # +120: all sessions are 2 hours

all_slot_ids = list(slot_info.keys())
print(f"Built {len(all_slot_ids)} distinct weekly slots from the actual data "
      f"(vs. 35 in the original fixed slots.csv grid).")


Built 29 distinct weekly slots from the actual data (vs. 35 in the original fixed slots.csv grid).


In [4]:
# ---- data-quality checks, printed rather than silently patched ----
missing_profile_tutors = sorted({
    r["TUTOR_ID"] for r in schedule_rows if r["TUTOR_ID"] not in tutor_to_profile_raw
})
if missing_profile_tutors:
    print(f"WARNING: {len(missing_profile_tutors)} tutors have no entry in "
          f"tutors_anonymized.csv, defaulting them to P6 (no preference): "
          f"{missing_profile_tutors}")

used_rooms = {r["ROOM_ID"] for r in schedule_rows}
unknown_capacity_rooms = sorted(used_rooms - set(room_capacity.keys()))
if unknown_capacity_rooms:
    n_affected_sessions = sum(1 for r in schedule_rows if r["ROOM_ID"] in unknown_capacity_rooms)
    print(f"WARNING: {len(unknown_capacity_rooms)} rooms used in this data have no "
          f"capacity on file ({n_affected_sessions} sessions affected): "
          f"{unknown_capacity_rooms}")
    print("         -> H4 capacity checks are skipped (not assumed passing) for these "
          "rooms, and they are excluded from the assignable room pool the optimiser "
          "can move sessions INTO (they can still be moved OUT of).")

def tutor_profile(tutor_id):
    """P6 (no preference) is the documented fallback for tutors missing from
    tutors_anonymized.csv -- see the warning above."""
    return tutor_to_profile_raw.get(tutor_id, "P6")

# ---- assignable room pool: only rooms with a verified capacity ----
all_room_ids = sorted(set(room_capacity.keys()))


In [5]:
sessions = []
for r in schedule_rows:
    slot_id = slot_id_of[(r["DAY"], r["START_TIME"])]
    sessions.append({
        "semester": r["SEMESTER"],       # NOT a decision variable -- fixed, identifies the term
        "course": r["COURSE_ID"],
        "class_id": r["CLASS_ID"],
        "tutor": r["TUTOR_ID"],          # fixed input, never reassigned
        "enrolled": int(r["ENROLLED_STUDENTS"]),
        "slot": slot_id,                 # DECISION VARIABLE: SA reassigns this
        "room": r["ROOM_ID"],            # DECISION VARIABLE: SA reassigns this
    })

def interval(s):
    """(day, start_minute, end_minute) for a session, looked up from its current slot."""
    return slot_info[s["slot"]]

def building_of(s):
    return room_building.get(s["room"])  # None if room capacity/building is unknown

print(f"Working dataset: {len(sessions)} sessions, "
      f"{len(set(s['tutor'] for s in sessions))} tutors, "
      f"{len(set(s['course'] for s in sessions))} courses, "
      f"{len(set(s['class_id'] for s in sessions))} distinct class IDs, "
      f"{len(set(s['room'] for s in sessions))} rooms as currently scheduled.")


Working dataset: 556 sessions, 79 tutors, 22 courses, 268 distinct class IDs, 40 rooms as currently scheduled.


## 2. Hard constraints (H1–H7)

Every function below checks exactly one row of the constraints table in
`Objective_Function_and_Constraints.docx`, Section 2. Each one returns a **count of
violations**; `count_hard_violations_breakdown()` at the end collects all seven into
one dictionary so you can see precisely which constraint is contributing what, instead
of a single opaque number.

### H1 — Assignment completeness
*Every session is assigned to exactly one (slot, room) pair.*

This is enforced by the data structure itself: every entry in `sessions` always has
exactly one `"slot"` and one `"room"` field, so there is nothing to actively check.
Included here only so H1 is visible in the code the same way it is visible in the
docx table, not silently skipped.

In [6]:
def h1_assignment_completeness(sessions_state):
    """Always 0 -- structurally guaranteed by the one-slot-one-room representation.
    Kept as a real function (not just a comment) so it shows up in the breakdown."""
    return 0


### H2 — No room double-booking, H3 — No tutor double-booking

Both are checked via true **time-interval overlap**, not exact slot-ID equality. This
matters because the real slot grid is irregular (Section 1): a 09:00–11:00 session and
an 08:00–10:00 session have different slot IDs but still overlap for an hour. Exact-ID
matching (as in the small-slice draft, which only had a uniform 2-hour grid) would miss
that. Since every session is a fixed 120 minutes, two sessions overlap iff their start
times are less than 120 minutes apart.

**Scoped per semester** (see the note in Section 0): a room or tutor used at the same
weekly day/time in two *different* semesters is not a real clash, since those semesters
never literally coexist. Getting this wrong the first time inflated the violation count
from 599 to 3706 by counting every room/tutor that simply gets reused, term after term,
as if it were double-booked.

In [7]:
def h2_h3_clashes(sessions_state):
    """Returns (room_clashes, tutor_clashes)."""
    room_groups = defaultdict(list)   # (room, semester, day) -> [(start,end), ...]
    tutor_groups = defaultdict(list)  # (tutor, semester, day) -> [(start,end), ...]

    for s in sessions_state:
        day, st, en = interval(s)
        room_groups[(s["room"], s["semester"], day)].append((st, en))
        tutor_groups[(s["tutor"], s["semester"], day)].append((st, en))

    def count_overlaps(groups):
        total = 0
        for key, ivals in groups.items():
            ivals.sort()
            for i in range(len(ivals)):
                for j in range(i + 1, len(ivals)):
                    if ivals[j][0] < ivals[i][1]:   # next start is before this one ends
                        total += 1
                    else:
                        break  # sorted by start -- no further overlaps possible past here
        return total

    return count_overlaps(room_groups), count_overlaps(tutor_groups)


### H4 — Room capacity / eligibility
*Enrolled students must not exceed room capacity.*

Rooms with no capacity on file (see the warning printed in Section 1) are **skipped**,
not assumed to pass -- `unknown_skipped` reports how many sessions couldn't be checked
so this limitation stays visible in every run rather than being silently absorbed into
a clean-looking zero.

In [8]:
def h4_capacity_violations(sessions_state):
    """Returns (violation_count, unknown_skipped_count)."""
    violations = 0
    unknown_skipped = 0
    for s in sessions_state:
        cap = room_capacity.get(s["room"])
        if cap is None:
            unknown_skipped += 1
            continue
        if cap < s["enrolled"]:
            violations += 1
    return violations, unknown_skipped


### H5 — No cross-campus back-to-back
*A tutor cannot be assigned rooms in different buildings in consecutive periods,
grounded in Zhu et al. (2026)'s hard constraint against same-day cross-campus travel.*

"Consecutive" is interpreted as **zero gap** between two sessions on the same day for
the same tutor (end of one exactly equals start of the next) -- i.e. no time at all to
physically travel between buildings. Scoped per (tutor, semester, day), for the same
reason as H2/H3.

In [9]:
def h5_cross_campus(sessions_state):
    tutor_day = defaultdict(list)  # (tutor, semester, day) -> [(start, end, building), ...]
    for s in sessions_state:
        day, st, en = interval(s)
        tutor_day[(s["tutor"], s["semester"], day)].append((st, en, building_of(s)))

    total = 0
    for key, items in tutor_day.items():
        items.sort(key=lambda x: (x[0], x[1]))  # sort by start, end (building may be None)
        for i in range(len(items) - 1):
            st1, en1, b1 = items[i]
            st2, en2, b2 = items[i + 1]
            if st2 == en1 and b1 is not None and b2 is not None and b1 != b2:
                total += 1
    return total


### H6 — No student conflict by year/major (curriculum conflict) — NEW, placeholder

Formally: define curricula (year+major cohorts) as groups of courses that potentially
share students; two courses in the same curriculum must not be scheduled in the same
period. Grounded in the Curriculum-Based Course Timetabling (CB-CTT) model (Ceschia, Di
Gaspero & Schaerf 2022; the same model underlying Mühlenthaler & Wanka).

**This cannot be checked yet.** `schedule_cleaned_it.csv` has `COURSE_ID`, `CLASS_ID`,
`SECTION`, but nothing that groups courses into a shared-student cohort (no year/major
field). Rather than guessing at a mapping, this returns 0 and is wired into the cost
function exactly where a real implementation would plug in once that mapping exists.

In [10]:
def h6_curriculum_conflict(sessions_state):
    """PLACEHOLDER. Needs a course -> curriculum (year+major cohort) mapping that does
    not exist in the current cleaned data. Do not invent one -- returns 0 until a real
    mapping is supplied, at which point this function is where it gets implemented:
    group sessions by curriculum, then count same-period pairs across DIFFERENT courses
    within the same curriculum group (mirroring h2_h3_clashes' overlap logic above, but
    keyed on curriculum membership instead of room/tutor identity)."""
    return 0


### H7 — Same-class day-distinctness — NEW
*A class's multiple weekly sessions must fall on different days.*

Documented in the small-slice draft as an unhandled simplification -- this implements it.
`CLASS_ID` is **not** unique across semesters (56 class IDs recur across 24A/24B/25A/25B
in this data, presumably the same recurring course-section offered term after term), so
this is scoped per `(semester, class_id)`, not by `class_id` alone.

In [11]:
def h7_same_class_day_distinct(sessions_state):
    groups = defaultdict(list)  # (semester, class_id) -> [day, day, ...]
    for s in sessions_state:
        day, st, en = interval(s)
        groups[(s["semester"], s["class_id"])].append(day)

    total = 0
    for key, days in groups.items():
        day_counts = defaultdict(int)
        for d in days:
            day_counts[d] += 1
        # each extra session beyond the first on the same day is one violation
        total += sum(c - 1 for c in day_counts.values() if c > 1)
    return total


### H1–H7 combined

`count_hard_violations_breakdown()` returns every constraint's count individually (so
you can see exactly which one is driving the total), and `count_hard_violations()`
sums them for the cost function.

In [12]:
def count_hard_violations_breakdown(sessions_state):
    room_clashes, tutor_clashes = h2_h3_clashes(sessions_state)
    capacity_viol, capacity_unknown = h4_capacity_violations(sessions_state)
    return {
        "H1_assignment_completeness": h1_assignment_completeness(sessions_state),
        "H2_room_clash": room_clashes,
        "H3_tutor_clash": tutor_clashes,
        "H4_capacity": capacity_viol,
        "H4_capacity_unknown_skipped": capacity_unknown,  # informational, not in the total
        "H5_cross_campus": h5_cross_campus(sessions_state),
        "H6_curriculum_conflict": h6_curriculum_conflict(sessions_state),
        "H7_same_class_same_day": h7_same_class_day_distinct(sessions_state),
    }

def count_hard_violations(sessions_state):
    b = count_hard_violations_breakdown(sessions_state)
    return sum(v for k, v in b.items() if k != "H4_capacity_unknown_skipped")

def is_feasible(sessions_state):
    return count_hard_violations(sessions_state) == 0


## 3. Logical constraint (L1)

*Maximum idle-gap cap: no tutor should have more than 4 hours of idle time between two
sessions on the same day.* This is new, distinct from both the hard constraints above
(it's undesirable, not infeasible) and the soft preferences below (it applies to every
tutor regardless of personal profile, it isn't a matter of taste). Scoped per
(tutor, semester, day), same reasoning as H2/H3/H5.

In [13]:
def l1_idle_cap_violations(sessions_state):
    tutor_day = defaultdict(list)  # (tutor, semester, day) -> [(start, end), ...]
    for s in sessions_state:
        day, st, en = interval(s)
        tutor_day[(s["tutor"], s["semester"], day)].append((st, en))

    total = 0
    for key, ivals in tutor_day.items():
        ivals.sort()
        for i in range(len(ivals) - 1):
            gap = ivals[i + 1][0] - ivals[i][1]
            if gap > IDLE_CAP_MINUTES:
                total += 1
    return total


## 4. Soft constraints (S1–S6) — per-tutor preference penalties

Each tutor is mapped to one of six profiles (`profiles.csv`). These are the same
penalty definitions as the small-slice draft, unchanged, just scoped per
(semester, day) now that multiple semesters are in play, so a tutor's "compact week"
or "back-to-back" pattern is judged within one term's actual week, not smeared across
several different terms' weeks at once.

| ID | Profile | Penalty |
|---|---|---|
| S1 | P1 Morning-only | +1 per session at/after 12:00 |
| S2 | P2 Night owl | +1 per session before 12:00 |
| S3 | P3 Compact week | number of distinct (semester, day) combinations used |
| S4 | P4 Prefers idle time | count of back-to-back (zero-gap) session pairs |
| S5 | P5 Dislikes idle time | count of idle gaps between sessions |
| S6 | P6 No preference | always 0 |

In [14]:
def is_morning(start_minute):
    return start_minute < 12 * 60

def tutor_penalty(tutor_id, ivals_by_semester_day):
    """ivals_by_semester_day: dict of (semester, day) -> [(start, end), ...] for one tutor,
    across every session they have in the selected semesters."""
    profile = tutor_profile(tutor_id)
    penalty = 0

    if profile == "P1":  # S1: morning-only
        for key, ivals in ivals_by_semester_day.items():
            penalty += sum(1 for st, en in ivals if not is_morning(st))

    elif profile == "P2":  # S2: night owl
        for key, ivals in ivals_by_semester_day.items():
            penalty += sum(1 for st, en in ivals if is_morning(st))

    elif profile == "P3":  # S3: compact week -- fewer distinct teaching days is better.
        penalty += len(ivals_by_semester_day)  # count of (semester, day) groups used

    elif profile == "P4":  # S4: prefers idle time -- penalise back-to-back pairs
        for key, ivals in ivals_by_semester_day.items():
            ivals_sorted = sorted(ivals)
            for i in range(len(ivals_sorted) - 1):
                if ivals_sorted[i + 1][0] == ivals_sorted[i][1]:  # zero gap
                    penalty += 1

    elif profile == "P5":  # S5: dislikes idle time -- penalise every gap
        for key, ivals in ivals_by_semester_day.items():
            ivals_sorted = sorted(ivals)
            for i in range(len(ivals_sorted) - 1):
                gap = ivals_sorted[i + 1][0] - ivals_sorted[i][1]
                if gap > 0:
                    penalty += 1

    # P6: no preference -- penalty stays 0

    return SOFT_WEIGHTS[profile] * penalty


def per_tutor_violations(sessions_state):
    """V_t for every tutor t: total weighted soft-preference violation,
    combined across all selected semesters. This is the V_t in
    V_t = sum_k w_k * Penalty_t,k(s) from the docx, Section 1.1."""
    tutor_groups = defaultdict(lambda: defaultdict(list))
    for s in sessions_state:
        day, st, en = interval(s)
        tutor_groups[s["tutor"]][(s["semester"], day)].append((st, en))

    V = {}
    for t, ivals_by_semester_day in tutor_groups.items():
        V[t] = tutor_penalty(t, ivals_by_semester_day)
    return V


## 5. Fairness function F(V_1,...,V_T)

Primary form: Jain's fairness index, inverted per Mühlenthaler & Wanka so that zero
violation reads as maximally fair rather than the reverse. Secondary form: max-min
(lexicographic), included for the H1 comparison but **not** used in the SA cost
function below -- see docx Section 3.3, this choice is flagged as open, not locked in.

In [15]:
def jain_fairness(V_dict):
    """J(V') in [1/T, 1]. 1 = every tutor carries an equal share of violation."""
    vals = list(V_dict.values())
    T = len(vals)
    if T == 0:
        return 1.0
    v_max = max(vals)
    shifted = [v_max - v for v in vals]   # V'_t = V_max - V_t, per docx Section 1.2
    s1 = sum(shifted)
    s2 = sum(v * v for v in shifted)
    if s2 == 0:
        return 1.0  # every tutor has identical violation (including all-zero) -> perfectly fair
    return (s1 ** 2) / (T * s2)

def fairness_cost(V_dict):
    """F(s) = 1 - J(V'). This is what the SA cost function minimises."""
    return 1 - jain_fairness(V_dict)

def maxmin_sorted_key(V_dict):
    """Secondary/comparison fairness form (docx Section 1.2): the sorted-descending
    violation vector, for lexicographic comparison between two schedules. Not used by
    the SA cost function -- reported alongside Jain's index for the future H1 test."""
    return tuple(sorted(V_dict.values(), reverse=True))


## 6. Full objective / cost function

`Total(s) = alpha * H(s) + beta * L(s) + F(V_1(s),...,V_T(s))`, per docx Section 1.3.

In [16]:
def cost_function(sessions_state):
    """Returns (total_cost, hard_breakdown_dict, logical_count, V_dict, fairness_F)."""
    hard_breakdown = count_hard_violations_breakdown(sessions_state)
    hard_total = sum(v for k, v in hard_breakdown.items() if k != "H4_capacity_unknown_skipped")
    logical_total = l1_idle_cap_violations(sessions_state)
    V = per_tutor_violations(sessions_state)
    F = fairness_cost(V)

    total = HARD_PENALTY * hard_total + LOGICAL_PENALTY * logical_total + F
    return total, hard_breakdown, logical_total, V, F


## 7. Baseline diagnostics — the real, current schedule before any optimisation

This prints every constraint's contribution separately, exactly what was asked for:
where each hard constraint is being violated, then the logical constraint, then the
soft/fairness picture.

In [17]:
def print_diagnostics(label, sessions_state):
    total, hard_breakdown, logical_total, V, F = cost_function(sessions_state)
    print(f"--- {label} ---")
    print("Hard constraints:")
    for k, v in hard_breakdown.items():
        print(f"    {k:32s}: {v}")
    hard_total = sum(v for k, v in hard_breakdown.items() if k != "H4_capacity_unknown_skipped")
    print(f"    {'TOTAL hard violations':32s}: {hard_total}")
    print(f"Logical (L1, idle > {IDLE_CAP_MINUTES} min) violations: {logical_total}")
    print(f"Soft/fairness:")
    print(f"    Raw summed violation across tutors (V_t sum) : {sum(V.values())}")
    print(f"    Jain's fairness index J(V')                  : {1 - F:.4f}  (1 = perfectly fair)")
    print(f"    Fairness cost F = 1 - J                      : {F:.4f}")
    print(f"    Worst-off tutor's V_t (max-min comparison)   : {max(V.values()) if V else 0}")
    print(f"TOTAL COST = alpha*H + beta*L + F                : {total:.2f}")
    print()
    return total

baseline_cost = print_diagnostics("BASELINE (current, un-optimised schedule)", sessions)


--- BASELINE (current, un-optimised schedule) ---
Hard constraints:
    H1_assignment_completeness      : 0
    H2_room_clash                   : 16
    H3_tutor_clash                  : 15
    H4_capacity                     : 4
    H4_capacity_unknown_skipped     : 0
    H5_cross_campus                 : 19
    H6_curriculum_conflict          : 0
    H7_same_class_same_day          : 0
    TOTAL hard violations           : 54
Logical (L1, idle > 240 min) violations: 0
Soft/fairness:
    Raw summed violation across tutors (V_t sum) : 228
    Jain's fairness index J(V')                  : 0.7423  (1 = perfectly fair)
    Fairness cost F = 1 - J                      : 0.2577
    Worst-off tutor's V_t (max-min comparison)   : 6
TOTAL COST = alpha*H + beta*L + F                : 54000.26



## 8. Simulated annealing

Neighbour move, acceptance rule, and cooling schedule are unchanged from the small-slice
draft (Kirkpatrick, Gelatt & Vecchi 1983): pick one random session, reassign it to a
random (slot, room) pair, accept via the Metropolis criterion, cool geometrically. Only
the cost function body changed, to add the logical term and swap the raw soft sum for
the fairness function.

In [18]:
def sa_optimise(initial_sessions, max_iterations, T0, cooling):
    current = [dict(s) for s in initial_sessions]
    current_cost, *_ = cost_function(current)
    best = [dict(s) for s in current]
    best_cost = current_cost

    T = T0
    accepted = 0
    history = []

    for it in range(1, max_iterations + 1):
        idx = random.randrange(len(current))
        old_slot, old_room = current[idx]["slot"], current[idx]["room"]

        current[idx]["slot"] = random.choice(all_slot_ids)
        current[idx]["room"] = random.choice(all_room_ids)

        new_cost, *_ = cost_function(current)
        delta = new_cost - current_cost

        if delta <= 0 or random.random() < math.exp(-delta / max(T, 1e-9)):
            current_cost = new_cost
            accepted += 1
            if current_cost < best_cost:
                best_cost = current_cost
                best = [dict(s) for s in current]
        else:
            current[idx]["slot"], current[idx]["room"] = old_slot, old_room

        history.append(current_cost)
        T *= cooling

    return best, best_cost, accepted, history


## 9. Run

`MAX_ITERATIONS` (Section 0) is a starting point sized for this dataset's search space
(tens of thousands of sessions x slots x rooms), not a tuned final value -- formal
tuning of iteration count and cooling schedule against runtime/quality is separate,
not-yet-done work (matching the O4 status already flagged in the mid-review).

In [19]:
t0 = time.time()
best_sessions, best_cost, accepted, history = sa_optimise(
    sessions, max_iterations=MAX_ITERATIONS, T0=T0, cooling=COOLING
)
elapsed = time.time() - t0

print(f"Iterations: {MAX_ITERATIONS}, accepted moves: {accepted}, "
      f"runtime: {elapsed:.1f}s\n")
final_cost = print_diagnostics("SIMULATED ANNEALING RESULT", best_sessions)

print(f"Combined-cost reduction: {baseline_cost - final_cost:.2f} "
      f"({100 * (baseline_cost - final_cost) / baseline_cost:.1f}%)")
print(f"Final solution fully hard-feasible: {is_feasible(best_sessions)}")


Iterations: 10000, accepted moves: 2746, runtime: 16.3s

--- SIMULATED ANNEALING RESULT ---
Hard constraints:
    H1_assignment_completeness      : 0
    H2_room_clash                   : 0
    H3_tutor_clash                  : 0
    H4_capacity                     : 0
    H4_capacity_unknown_skipped     : 0
    H5_cross_campus                 : 0
    H6_curriculum_conflict          : 0
    H7_same_class_same_day          : 0
    TOTAL hard violations           : 0
Logical (L1, idle > 240 min) violations: 1
Soft/fairness:
    Raw summed violation across tutors (V_t sum) : 189
    Jain's fairness index J(V')                  : 0.9385  (1 = perfectly fair)
    Fairness cost F = 1 - J                      : 0.0615
    Worst-off tutor's V_t (max-min comparison)   : 8
TOTAL COST = alpha*H + beta*L + F                : 50.06

Combined-cost reduction: 53950.20 (99.9%)
Final solution fully hard-feasible: True


## 10. Convergence plot

In [20]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 4))
    plt.plot(history)
    plt.xlabel("Iteration")
    plt.ylabel("Current cost (alpha*H + beta*L + F)")
    plt.title(f"SA convergence -- semesters {sorted(SEMESTERS)}, {len(sessions)} sessions")
    plt.tight_layout()
    plt.savefig("sa_convergence.png", dpi=150)
    plt.show()
except ImportError:
    print("matplotlib not installed -- skip this cell or `pip install matplotlib`.")


matplotlib not installed -- skip this cell or `pip install matplotlib`.


## 11. Reusing this notebook for other semesters

To run this on a different combination of terms: change `SEMESTERS` in Section 0 to any
subset of `{"23A", "23B", "24A", "24B", "25A", "25B"}` and re-run the whole notebook top
to bottom. Nothing else needs editing, the slot grid, data-quality checks, and every
constraint function all rebuild themselves from whatever `SEMESTERS` contains.

**Open items carried over from the objective-function document, still true here:**
- H6 (curriculum conflict) is a placeholder until a course-to-curriculum mapping exists.
- Jain's index vs. max-min as the *headline* fairness metric for RQ2/H1 is not locked in
  (both are computed and reported every run, Section 7, so the comparison is ready
  whenever that decision is made).
- `LOGICAL_PENALTY` (beta) and the per-profile `SOFT_WEIGHTS` (w_k) are placeholders,
  not calibrated against the tutor-preference survey.
